<a href="https://colab.research.google.com/github/edwardoughton/GeoAI/blob/main/07_01_ggs590_geoai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Week 7 GeoAI: AI-supported automation for GIS workflows (Part 2)

This week moves from a one-off notebook workflow to a reproducible geospatial pipeline.

We keep the LULC framing from Week 6 and add environment control, reliability gates, and artifact outputs.


# Learning objectives

By the end of this class, students should be able to:
- Explain why virtual environments are essential for reproducible GeoAI workflows
- Create and activate a `venv` in VS Code, then export and recreate dependencies
- Refactor a GIS workflow into reusable pipeline functions
- Add quality gates before model fitting
- Produce auditable run artifacts (metrics, matrices, logs, outputs)


# Class structure

1. Environment setup and reproducibility (venv)
2. Pipeline design and validation gates
3. Modeling and diagnostics
4. Intermediate spatial visualization checks
5. Artifact writing and reproducible reruns


## Phase 0: Virtual environments in VS Code (venv)

### Why we do this

- Avoid package conflicts between projects
- Ensure everyone in class runs the same dependency set
- Make workflows portable from one machine to another
- Support grading and collaboration with reproducible environments

In short: a model result is only trustworthy if the environment is reproducible.


### VS Code workflow: create and activate a venv

In the integrated terminal from your project root:

```bash
python3 -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip
```

Then in VS Code:
1. Open Command Palette (`Cmd+Shift+P` on macOS)
2. Select `Python: Select Interpreter`
3. Choose `.venv/bin/python`


### Install and export dependencies

In notebook cells, prefer `%pip install ...` so installs target the active kernel environment.

Export exact package versions:

```bash
pip freeze > requirements.txt
```

Recreate elsewhere:

```bash
python3 -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt
```


### Exercise 0 (10 min): environment reproducibility check

1. Activate your `venv`
2. Verify interpreter path with `python -c "import sys; print(sys.executable)"`
3. Export `requirements.txt`
4. Pair with another student and compare 3 key package versions

Critical reflection: which package mismatch would most likely break geospatial workflows first, and why?


## Setup

Run the install cell only if packages are missing in your selected kernel.


In [ ]:
# Optional installs (uncomment if needed)
# %pip install numpy pandas matplotlib seaborn scikit-learn


In [ ]:
import json
import warnings
from dataclasses import dataclass, asdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, balanced_accuracy_score
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
np.random.seed(42)


### Function Guide: Config and run paths

Keep high-impact settings in one config class.


In [ ]:
@dataclass
class PipelineConfig:
    run_name: str = "week7_demo_run"
    output_root: str = "outputs/week7"
    random_state: int = 42
    test_size: float = 0.25
    n_estimators: int = 300
    max_depth: int = 20
    min_class_count: int = 100
    max_samples_per_class: int = 2500
    leakage_distance_hint_m: float = 30.0


cfg = PipelineConfig()
cfg


### Exercise 1 (10 min): config design critique

1. Add one new config parameter related to reproducibility or trust.
2. Explain where in the pipeline it should be used.

Coding practice: implement your parameter in one downstream function.


### Function Guide: Artifact management and logging


In [ ]:
def utc_now_iso():
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()


def prepare_run_dirs(config):
    root = Path(config.output_root) / config.run_name
    dirs = {
        "root": root,
        "figures": root / "figures",
        "tables": root / "tables",
        "rasters": root / "rasters",
        "logs": root / "logs",
    }
    for p in dirs.values():
        p.mkdir(parents=True, exist_ok=True)
    return dirs


def write_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)


def append_log(message, log_path):
    with open(log_path, "a", encoding="utf-8") as f:
        f.write("[{}] {}\n".format(utc_now_iso(), message))


## Phase 1: Build synthetic imagery + labels for intermediate map checks

This keeps class runnable without remote data dependencies while still teaching geospatial QA logic.


In [ ]:
def make_synthetic_scene(h=180, w=220, random_state=42):
    rng = np.random.default_rng(random_state)
    y, x = np.mgrid[0:h, 0:w]

    hill = np.exp(-(((x - 70) ** 2) + ((y - 60) ** 2)) / 2800)
    valley = np.exp(-(((x - 150) ** 2) + ((y - 130) ** 2)) / 3400)
    noise = rng.normal(0, 0.03, size=(h, w))

    red = (0.25 + 0.35 * hill + 0.10 * noise).clip(0, 1)
    nir = (0.30 + 0.45 * hill - 0.25 * valley + 0.10 * noise).clip(0, 1)
    green = (0.22 + 0.20 * hill + 0.15 * valley + 0.10 * noise).clip(0, 1)
    blue = (0.20 + 0.10 * valley + 0.10 * noise).clip(0, 1)

    ndvi = (nir - red) / (nir + red + 1e-6)

    labels = np.select(
        [ndvi > 0.25, blue > 0.28, red > 0.45],
        [1, 2, 3],
        default=4,
    )

    rgb = np.dstack([red, green, blue])

    return {
        "red": red,
        "green": green,
        "blue": blue,
        "nir": nir,
        "ndvi": ndvi,
        "labels": labels,
        "rgb": rgb,
    }


scene = make_synthetic_scene(random_state=cfg.random_state)
list(scene.keys())


### Stage A outputs: imagery diagnostics


In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(14, 4))

axs[0].imshow(scene["rgb"])
axs[0].set_title("Synthetic RGB")
axs[0].axis("off")

im1 = axs[1].imshow(scene["ndvi"], cmap="YlGn", vmin=-0.2, vmax=0.8)
axs[1].set_title("NDVI")
axs[1].axis("off")
plt.colorbar(im1, ax=axs[1], fraction=0.046, pad=0.04)

im2 = axs[2].imshow(scene["labels"], cmap="tab10", vmin=1, vmax=4)
axs[2].set_title("Label raster")
axs[2].axis("off")
plt.colorbar(im2, ax=axs[2], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()


### Stage B outputs: label alignment check (overlay)


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 5))
ax.imshow(scene["rgb"])
ax.imshow(scene["labels"], cmap="tab10", alpha=0.35, vmin=1, vmax=4)
ax.set_title("RGB + semi-transparent labels")
ax.axis("off")
plt.show()


### Exercise 2 (15 min): map QA and interpretation

1. Identify one area where labels look suspicious relative to imagery.
2. Propose one preprocessing step that might reduce this mismatch.
3. Edit visualization code to add a legend or custom colormap.

Critical reflection: what error could this mismatch create in model evaluation?


## Phase 2: Build a training table from raster-like arrays


In [ ]:
def scene_to_table(scene_dict):
    h, w = scene_dict["red"].shape
    y_idx, x_idx = np.mgrid[0:h, 0:w]
    return pd.DataFrame({
        "x": x_idx.ravel().astype(float),
        "y": y_idx.ravel().astype(float),
        "blue": scene_dict["blue"].ravel(),
        "green": scene_dict["green"].ravel(),
        "red": scene_dict["red"].ravel(),
        "nir": scene_dict["nir"].ravel(),
        "ndvi": scene_dict["ndvi"].ravel(),
        "label": scene_dict["labels"].ravel().astype(int),
    })


df = scene_to_table(scene)
df.head()


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 4))
sns.countplot(data=df, x="label", ax=ax)
ax.set_title("Class distribution before sampling")
plt.show()


### Function Guide: Sampling and validation gates


In [ ]:
def sample_per_class(df_in, label_col, max_per_class, random_state):
    sampled = []
    for cls, part in df_in.groupby(label_col):
        take = min(len(part), max_per_class)
        sampled.append(part.sample(take, random_state=random_state))
    return pd.concat(sampled, ignore_index=True)


def class_balance_report(y):
    counts = pd.Series(y).value_counts().sort_index()
    return pd.DataFrame({
        "class": counts.index,
        "count": counts.values,
        "share": counts.values / counts.values.sum(),
    })


def assert_min_class_count(y, min_class_count):
    counts = pd.Series(y).value_counts()
    too_small = counts[counts < min_class_count]
    if len(too_small) > 0:
        msg = ", ".join(["{}:{}".format(k, v) for k, v in too_small.items()])
        raise ValueError("Class count gate failed (min={}). Problem classes: {}".format(min_class_count, msg))


def leakage_proxy_check(train_xy, test_xy):
    if len(train_xy) == 0 or len(test_xy) == 0:
        return np.inf
    min_dist = np.inf
    for p in test_xy:
        d = np.sqrt(((train_xy - p) ** 2).sum(axis=1)).min()
        if d < min_dist:
            min_dist = d
    return float(min_dist)


In [ ]:
sampled_df = sample_per_class(
    df,
    label_col="label",
    max_per_class=cfg.max_samples_per_class,
    random_state=cfg.random_state,
)

report = class_balance_report(sampled_df["label"].to_numpy())
display(report)
assert_min_class_count(sampled_df["label"].to_numpy(), cfg.min_class_count)
print("Class-count gate passed.")


### Exercise 3 (10 min): gate tuning

1. Increase `min_class_count` until the gate fails.
2. Propose two valid responses when this happens.

Coding practice: implement one automatic response (for example, lower threshold with warning).


## Phase 3: Modeling with explicit controls


In [ ]:
feature_cols = ["blue", "green", "red", "nir", "ndvi"]

X = sampled_df[feature_cols].to_numpy()
y = sampled_df["label"].to_numpy()
XY = sampled_df[["x", "y"]].to_numpy()

X_train, X_test, y_train, y_test, XY_train, XY_test = train_test_split(
    X,
    y,
    XY,
    test_size=cfg.test_size,
    random_state=cfg.random_state,
    stratify=y,
)

dmin = leakage_proxy_check(XY_train, XY_test)
print("Leakage proxy minimum train-test distance: {:.3f}".format(dmin))


In [ ]:
rf = RandomForestClassifier(
    n_estimators=cfg.n_estimators,
    max_depth=cfg.max_depth,
    random_state=cfg.random_state,
    n_jobs=-1,
    class_weight="balanced_subsample",
)

rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

acc = accuracy_score(y_test, y_pred)
bacc = balanced_accuracy_score(y_test, y_pred)

print("Accuracy: {:.4f}".format(acc))
print("Balanced Accuracy: {:.4f}".format(bacc))


### Stage C outputs: confusion matrix and feature importance


In [ ]:
labels = sorted(np.unique(np.concatenate([y_test, y_pred])))
cm = confusion_matrix(y_test, y_pred, labels=labels, normalize="true")

fig, axs = plt.subplots(1, 2, figsize=(12, 4))

sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", cbar=False, ax=axs[0])
axs[0].set_title("Row-normalized confusion matrix")
axs[0].set_xlabel("Predicted")
axs[0].set_ylabel("Observed")

importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
importances.plot(kind="bar", ax=axs[1])
axs[1].set_title("Feature importance")
axs[1].set_ylabel("Importance")

plt.tight_layout()
plt.show()

print(classification_report(y_test, y_pred, digits=3))


### Exercise 4 (15 min): model diagnostics

1. Which class has the weakest recall, and what feature engineering might help?
2. Change one RF hyperparameter and rerun.
3. Record whether balanced accuracy improves or declines.

Critical reflection: did the metric change enough to justify the extra model complexity?


## Phase 4: Full-scene prediction maps (intermediate spatial checks)


In [ ]:
flat_features = np.column_stack([
    scene["blue"].ravel(),
    scene["green"].ravel(),
    scene["red"].ravel(),
    scene["nir"].ravel(),
    scene["ndvi"].ravel(),
])

pred_flat = rf.predict(flat_features)
pred_map = pred_flat.reshape(scene["labels"].shape)


In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 5))

axs[0].imshow(scene["rgb"])
axs[0].set_title("RGB imagery")
axs[0].axis("off")

axs[1].imshow(scene["labels"], cmap="tab10", vmin=1, vmax=4)
axs[1].set_title("Observed labels")
axs[1].axis("off")

axs[2].imshow(pred_map, cmap="tab10", vmin=1, vmax=4)
axs[2].set_title("Predicted map")
axs[2].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
mismatch = (pred_map != scene["labels"]).astype(int)

fig, axs = plt.subplots(1, 2, figsize=(10, 4))
axs[0].imshow(scene["rgb"])
axs[0].set_title("RGB reference")
axs[0].axis("off")

axs[1].imshow(mismatch, cmap="Reds", vmin=0, vmax=1)
axs[1].set_title("Prediction mismatch mask")
axs[1].axis("off")

plt.tight_layout()
plt.show()

print("Pixel mismatch rate: {:.3f}".format(mismatch.mean()))


### Exercise 5 (10 min): trust and map inspection

1. Where are map errors spatially clustered?
2. Add one new map layer that helps explain the error pattern (for example NDVI).

Coding practice: create a 2x2 panel with RGB, labels, predictions, and mismatches.


## Phase 5: Run artifact writing and reproducible reruns


In [ ]:
run_dirs = prepare_run_dirs(cfg)
log_path = run_dirs["logs"] / "run_log.txt"

append_log("Run started", log_path)
append_log("Model trained", log_path)

metrics = {
    "run_name": cfg.run_name,
    "timestamp_utc": utc_now_iso(),
    "accuracy": float(acc),
    "balanced_accuracy": float(bacc),
    "pixel_mismatch_rate": float(mismatch.mean()),
    "n_train": int(len(y_train)),
    "n_test": int(len(y_test)),
    "config": asdict(cfg),
}

write_json(metrics, run_dirs["tables"] / "metrics.json")
report.to_csv(run_dirs["tables"] / "class_balance.csv", index=False)
pd.DataFrame(cm, index=labels, columns=labels).to_csv(
    run_dirs["tables"] / "confusion_matrix_row_normalized.csv"
)

fig, ax = plt.subplots(1, 1, figsize=(6, 5))
ax.imshow(pred_map, cmap="tab10", vmin=1, vmax=4)
ax.set_title("Predicted map (artifact preview)")
ax.axis("off")
fig.savefig(run_dirs["figures"] / "predicted_map.png", dpi=200, bbox_inches="tight")
plt.show()

append_log("Artifacts written", log_path)
print("Artifacts saved under:", run_dirs["root"])


In [ ]:
display(pd.DataFrame([metrics]))

with open(run_dirs["logs"] / "run_log.txt", "r", encoding="utf-8") as f:
    print(f.read())


### Exercise 6 (15 min): portability and replication

1. Create `requirements.txt` from your active environment.
2. Change `cfg.run_name` and `cfg.random_state`, then rerun artifact cells.
3. Compare output metrics across runs.

Critical reflection: what minimum metadata should every run artifact include?


## Homework

- Apply this Week 7 structure to your Week 6 AOI workflow
- Keep all settings in `PipelineConfig`
- Add at least one additional quality gate
- Export `requirements.txt` and include environment recreation instructions
- Submit one short reflection on trust, reproducibility, and scaling
